In [ ]:
!pip install numpy pandas matplotlib seaborn scikit-learn imbalanced-learn xgboost shap streamlit

In [ ]:
from google.colab import files
files.upload()

In [ ]:
!mkdir -p ~/.kaggle
!cp kaggle.json ~/.kaggle/
!chmod 600 ~/.kaggle/kaggle.json
!kaggle datasets download -d mlg-ulb/creditcardfraud
!unzip creditcardfraud.zip

# Imports and Helpers

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report,confusion_matrix, roc_auc_score, precision_recall_fscore_support
from sklearn.ensemble import RandomForestClassifier
from imblearn.over_sampling import SMOTE
from imblearn.combine import SMOTEENN
from imblearn.over_sampling import ADASYN
import pickle
import warnings
warnings.filterwarnings('ignore')

def evaluate_model(model, X_test, y_test):
    print("✅ Evaluating model...\n")
    y_pred = model.predict(X_test)
    if hasattr(model, 'predict_proba'):
        y_proba = model.predict_proba(X_test)[:,1]
    else:
        y_proba = None

    # Metrics
    print("Classification Report:\n", classification_report(y_test, y_pred, digits=4))
    if y_proba is not None:
        print("ROC-AUC Score:", round(roc_auc_score(y_test, y_proba), 4))
    print("\nConfusion Matrix:")
    print(confusion_matrix(y_test, y_pred))



# Loading dataset

In [ ]:
path = 'creditcard.csv'
if not os.path.exists(path):
  raise FileNotFoundError("Place creditcard.csv in the working directory or upload to Colab Files.")
df = pd.read_csv(path)
print("Loaded:", df.shape)
display(df.head())

# Quick EDA

In [ ]:
print(df['Class'].value_counts())
print((df['Class'].value_counts(normalize=True)*100).round(4))
plt.figure(figsize=(6,4)); sns.countplot(x= 'Class', data=df); plt.title('Class counts'); plt.show()
plt.figure(figsize=(8,4)); sns.histplot(df['Amount'], bins=60, kde= True); plt.title('Transaction Amount'); plt.show()


# Preprocessing

In [ ]:
from sklearn.preprocessing import StandardScaler

data = df.copy()
scaler = StandardScaler()
data['Amount_Scaled'] = scaler.fit_transform(data['Amount'].values.reshape(-1,1))
if 'Time' in data.columns:
  data = data.drop(['Time','Amount'], axis=1)
X = data.drop('Class', axis=1)
y = data['Class']
print("Features:", X.shape, "Target:", y.shape)

# Train-test split

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, stratify=y, random_state=42)
print("Train:", X_train.shape, "Test:", X_test.shape)
print("Train class distribution:\n", y_train.value_counts())


# Baseline: RandomForest with class_weight

In [ ]:
rf = RandomForestClassifier(n_estimators=100, class_weight='balanced', random_state=42, n_jobs=-1)
rf.fit(X_train, y_train)
print("Baseline RF on test set:")
evaluate_model(rf, X_test, y_test)

# SMOTE resampling

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from imblearn.over_sampling import SMOTE

# Re-run SMOTE (fast version)
sm = SMOTE(random_state=42)
X_res, y_res = sm.fit_resample(X_train, y_train)
print("After SMOTE:", np.bincount(y_res))

# Lighter Random Forest model
rf_sm = RandomForestClassifier(
    n_estimators=50,    # fewer trees
    max_depth=10,       # limit tree depth
    max_features='sqrt',
    n_jobs=-1,
    random_state=42
)

print("Training smaller RF on SMOTE-resampled data...")
rf_sm.fit(X_res, y_res)
print("Done training. Now evaluating...")
evaluate_model(rf_sm, X_test, y_test)


# Trying SMOTE-ENN

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from imblearn.combine import SMOTEENN

# Using a subset for faster execution of the hybrid method
X_train_small, _, y_train_small, _ = train_test_split(
    X_train, y_train, train_size=0.25, stratify=y_train, random_state=42
)

print("Applying SMOTE-ENN...")
smenn = SMOTEENN(random_state=42)
X_smenn, y_smenn = smenn.fit_resample(X_train_small, y_train_small)
print("After SMOTE-ENN (subset):", np.bincount(y_smenn))

# Train a Random Forest on the cleaned, resampled data
rf_smenn = RandomForestClassifier(n_estimators=50, max_depth=10, n_jobs=-1, random_state=42)
rf_smenn.fit(X_smenn, y_smenn)

print("\n--- SMOTE-ENN Model Evaluation ---")
evaluate_model(rf_smenn, X_test, y_test)

In [ ]:
from sklearn.metrics import precision_recall_curve

# Get probability predictions
y_probs = xgb.predict_proba(X_test)[:, 1]
precision, recall, thresholds = precision_recall_curve(y_test, y_probs)

# Calculate F1 score for each threshold
f1_scores = 2 * (precision * recall) / (precision + recall + 1e-10)
best_threshold = thresholds[np.argmax(f1_scores)]

print(f'Best Threshold for F1-Score: {best_threshold:.4f}')

# Plot Precision-Recall Curve
plt.figure(figsize=(8, 5))
plt.plot(recall, precision, label='Precision-Recall curve')
plt.xlabel('Recall')
plt.ylabel('Precision')
plt.title('Precision-Recall Curve (XGBoost)')
plt.axvline(recall[np.argmax(f1_scores)], color='r', linestyle='--', label=f'Best F1 Threshold: {best_threshold:.2f}')
plt.legend()
plt.show()

In [ ]:
# Analyze correlations between features and the Class
plt.figure(figsize=(12, 10))
corr = df.corr()
sns.heatmap(corr[['Class']].sort_values(by='Class', ascending=False), annot=True, cmap='coolwarm', vmin=-1, vmax=1)
plt.title('Feature Correlation with Fraud (Class)')
plt.show()

### Final Project Summary
- **Resampling**: Used SMOTE and SMOTE-ENN to handle the extreme 0.17% class imbalance.
- **Model**: XGBoost with `scale_pos_weight` achieved high ROC-AUC and a strong balance of Precision/Recall.
- **Explainability**: SHAP identified the key features (like V17, V14, and V12) that most influence the fraud prediction.
- **Actionable Insight**: The threshold optimization allows the business to tune the sensitivity of the fraud detection system.

# XGBoost

In [ ]:
from xgboost import XGBClassifier

# Calculate imbalance ratio for scale_pos_weight
neg = sum(y_train == 0)
pos = sum(y_train == 1)
scale_pos_weight = neg / pos
print(f"Imbalance Ratio (scale_pos_weight): {scale_pos_weight:.2f}")

# Define and train the XGBoost model
xgb = XGBClassifier(
    n_estimators=200,
    max_depth=6,
    learning_rate=0.1,
    scale_pos_weight=scale_pos_weight,
    eval_metric='logloss',
    tree_method='hist',
    random_state=42,
    n_jobs=-1
)

print("Training XGBoost...")
xgb.fit(X_train, y_train)
print("\n--- XGBoost Model Evaluation ---")
evaluate_model(xgb, X_test, y_test)

In [ ]:
import pickle
with open("xgb_fraud_model.pkl", "wb") as f:
    pickle.dump(xgb, f)
print("✅ Model successfully saved as 'xgb_fraud_model.pkl'")

# Explainability (SHAP)

In [ ]:
import shap
import matplotlib.pyplot as plt

# Initialize SHAP
explainer = shap.TreeExplainer(xgb)

# Take a sample for explanation to save time
sample_size = 200
sample = X_test.sample(n=sample_size, random_state=42)
shap_values = explainer.shap_values(sample)

# Summary Plot
plt.figure(figsize=(10, 6))
shap.summary_plot(shap_values, sample, show=False)
plt.title("SHAP Feature Importance for XGBoost")
plt.show()

# Explain a specific fraud case (if any in sample) or just the first row
idx = 0
print(f"Explaining prediction for index: {sample.index[idx]}")
shap.initjs()
shap.force_plot(explainer.expected_value, shap_values[idx, :], sample.iloc[idx, :], matplotlib=True)

In [ ]:
%%writefile app.py
import streamlit as st
import pandas as pd
import numpy as np
import pickle
import os

# Load the saved model
@st.cache_resource
def load_model():
    with open('xgb_fraud_model.pkl', 'rb') as f:
        return pickle.load(f)

# Load a small sample of the actual data for the 'Fill Sample' feature
@st.cache_data
def get_sample_data():
    if os.path.exists('creditcard.csv'):
        # Load just 100 rows to keep it fast
        return pd.read_csv('creditcard.csv', nrows=100)
    return None

model = load_model()
df_sample = get_sample_data()

st.set_page_config(page_title="Fraud Detector", layout="wide")
st.title("💳 Credit Card Fraud Detection")

with st.sidebar:
    st.header("About the Features")
    st.info("Features V1-V28 are anonymized technical components (PCA) used to protect user privacy. In a production system, these would be calculated automatically from transaction metadata.")

    if st.button("💡 Fill with Random Sample"):
        if df_sample is not None:
            rand_row = df_sample.sample(1).iloc[0]
            st.session_state['v_values'] = rand_row
            st.rerun()

st.markdown("Enter details below or use the sidebar to load a sample.")

# Initialize session state for inputs if not present
if 'v_values' not in st.session_state:
    st.session_state['v_values'] = {f'V{i}': 0.0 for i in range(1, 29)}
    st.session_state['v_values']['Amount'] = 10.0

with st.form("prediction_form"):
    cols = st.columns(4)
    input_data = {}

    # Dynamic input creation
    for i in range(1, 29):
        v_name = f'V{i}'
        col_idx = (i-1) % 4
        val = float(st.session_state['v_values'].get(v_name, 0.0))
        input_data[v_name] = cols[col_idx].number_input(v_name, value=val)

    amount = st.number_input("Transaction Amount ($)", value=float(st.session_state['v_values'].get('Amount', 10.0)))
    input_data['Amount_Scaled'] = (amount - 88.3) / 250.1

    submit = st.form_submit_button("Run Fraud Analysis")

if submit:
    features = pd.DataFrame([input_data])
    v_cols = [f'V{i}' for i in range(1, 29)] + ['Amount_Scaled']
    features = features[v_cols]

    prediction = model.predict(features)[0]
    probability = model.predict_proba(features)[0][1]

    if prediction == 1:
        st.error(f"🚨 HIGH RISK DETECTED! Probability: {probability:.2%}")
    else:
        st.success(f"✅ Transaction Appears Safe. Probability of Fraud: {probability:.2%}")

In [ ]:
# 1. Install localtunnel
!npm install -g localtunnel

# 2. Run streamlit with CORS and XSRF disabled for easier Colab access
import subprocess
import time
import os

# Kill any existing streamlit processes to avoid port conflicts
!pkill streamlit
!pkill lt

# Run streamlit
# Using --server.fileWatcherType none to prevent some sync issues in Colab
subprocess.Popen(['streamlit', 'run', 'app.py', '--server.port', '8501', '--server.address', 'localhost', '--server.enableCORS', 'false', '--server.enableXsrfProtection', 'false', '--server.fileWatcherType', 'none'])

# Wait a few seconds for the server to start
time.sleep(5)

# 3. Get your public IP
print("Your Tunnel Password (IP) is:")
!curl ipv4.icanhazip.com

# 4. Create the tunnel
print("\nClick the link below, enter the IP above, and click 'Click to Submit':")
!npx localtunnel --port 8501